# E3 — Замкнутый MPC-бенчмарк (ядро)

Сравнение регуляторов по **EPI** и нарушениям на тест-сезоне (Ростов 2020): `rule_based`, `sindy_mpc` (рецепт из E2), `grey_box_mpc` (редуцированная линейная физика), `nn_mpc`, `ppo`, `sac`, и **`oracle_mpc`** (CEM по истинной модели симулятора `env.F`). Источник дисперсии — **сид в сборе train** (суррогат переобучается на каждом сиде; ≥10 прогонов в article-grade). Статистика — Wilcoxon + Holm + bootstrap CI. Тяжёлые регуляторы (oracle, RL) — на сокращённом числе сидов (по бюджету CPU).

In [1]:
import os, sys, json, time, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath("."))
import article_experiment_utils as U
import protocol_config as P

# FAST_MODE smoke (tiny data) vs article-grade. Toggle via env var ARTICLE_FAST=0.
FAST_MODE = os.environ.get("ARTICLE_FAST", "1") == "1"
pc = P.DEFAULT.resolved(FAST_MODE)
RES = U.results_dir()
ECON = P.read_env_economics(pc.location)
CORR, PRICES = ECON["corridors"], ECON["prices"]
print("FAST_MODE", FAST_MODE, "| seeds", tuple(pc.seeds), "| budgets", pc.budgets_days,
      "| n_days_train/test", pc.n_days_train, pc.n_days_test)

FAST_MODE True | seeds (0, 1) | budgets (1, 3, 5) | n_days_train/test 5 3


## Замороженный рецепт E2 и параметры прогона

In [2]:
recipe = json.loads((RES / "recipe_frozen.json").read_text(encoding="utf-8"))["recipe"] \
    if (RES / "recipe_frozen.json").exists() else {"feature_variant": "physics", "library_degree": 1, "optimizer": "stlsq", "denoise": "savgol"}
test_scen = pc.test_scenario(); TEST_START = test_scen["start_date"]; N_TEST = pc.n_days_test
train_sc = pc.train_scenarios()[0]
seeds = list(pc.seeds)
oracle_seeds = seeds[:1]                       # oracle is the expensive controller
rl_seeds = seeds[: (1 if FAST_MODE else 3)]    # RL training budget-limited
print("recipe", recipe, "| seeds", seeds, "| oracle_seeds", oracle_seeds, "| rl_seeds", rl_seeds)

recipe {'feature_variant': 'physics', 'library_degree': 1, 'optimizer': 'stlsq', 'denoise': 'savgol'} | seeds [0, 1] | oracle_seeds [0] | rl_seeds [0]


## Многосидовый замкнутый прогон (суррогаты переобучаются на каждом сиде)

In [3]:
records = []; traj0 = None
for s in seeds:
    cfg_s = pc.cfg_for(test_scen, seed=s)
    train_s = U.collect_rule_based_dataset(pc.cfg_for(train_sc, seed=s), n_days=pc.n_days_train, prbs_scale=0.3)
    b_sindy = U.fit_sindy(train_s, feature_variant=recipe["feature_variant"], library_degree=recipe["library_degree"],
                          optimizer=recipe["optimizer"], denoise=recipe["denoise"], period=float(pc.period), metadata={"label": "sindy_mpc"})
    b_grey = U.fit_sindy(train_s, feature_variant="physics_no_cross", library_degree=1, threshold=1e-6, period=float(pc.period), metadata={"label": "grey_box_mpc"})
    nn = U.fit_nn_surrogate(train_s, feature_variant="physics", hidden_sizes=[64, 64], epochs=(60 if FAST_MODE else 300), period=float(pc.period), metadata={"label": "nn_mpc"})
    rollouts = {
        "rule_based": U.rollout_rule_based(cfg_s, n_days=N_TEST, start_date=TEST_START, noise_scale=0.0, seed=s),
        "sindy_mpc": U.rollout_mpc(b_sindy, cfg_s, n_days=N_TEST, start_date=TEST_START),
        "grey_box_mpc": U.rollout_mpc(b_grey, cfg_s, n_days=N_TEST, start_date=TEST_START),
        "nn_mpc": U.rollout_mpc_nn(nn, cfg_s, n_days=N_TEST, start_date=TEST_START, horizon=pc.horizon),
    }
    if s in rl_seeds:
        for algo in ("ppo", "sac"):
            mdl = U.train_rl(algo, pc.cfg_for(train_sc, seed=s), pc.rl_train_steps, train_start_date=train_sc["start_date"], seed=s)
            rollouts[algo] = U.rollout_rl(mdl, cfg_s, n_days=N_TEST, start_date=TEST_START, label=algo)
    if s in oracle_seeds:
        rollouts["oracle_mpc"] = U.rollout_oracle_mpc(cfg_s, n_days=N_TEST, start_date=TEST_START,
                                                      n_samples=(32 if FAST_MODE else 64), n_iters=(2 if FAST_MODE else 3))
    for name, dfr in rollouts.items():
        mm = U.epi_metrics(dfr, corridors=CORR, prices=PRICES); mm.update({"method": name, "seed": s}); records.append(mm)
    if traj0 is None:
        traj0 = rollouts
    print("seed", s, "done; methods:", list(rollouts.keys()))
seeded = pd.DataFrame(records)
U.save_table(seeded, RES / "tables" / "e3_seeded.csv")
print("records", len(seeded))

seed 0 done; methods: ['rule_based', 'sindy_mpc', 'grey_box_mpc', 'nn_mpc', 'ppo', 'sac', 'oracle_mpc']


seed 1 done; methods: ['rule_based', 'sindy_mpc', 'grey_box_mpc', 'nn_mpc']
records 11


## Главная таблица mean±std, разрыв до oracle, и статистика vs rule_based

In [4]:
agg = seeded.groupby("method").agg(epi_mean=("epi", "mean"), epi_std=("epi", "std"),
        viol_mean=("violation_steps_total", "mean"), Tcorr=("t_in_in_corridor_pct", "mean"),
        n=("epi", "size")).reset_index().sort_values("epi_mean", ascending=False)
if "oracle_mpc" in seeded.method.values:
    oracle_epi = seeded[seeded.method == "oracle_mpc"]["epi"].mean()
    agg["gap_to_oracle"] = oracle_epi - agg["epi_mean"]
U.save_table(agg, RES / "tables" / "e3_main_table.csv")
stats = U.paired_stats(seeded, "epi", baseline="rule_based")
U.save_table(stats, RES / "tables" / "e3_stats_vs_rulebased.csv")
display(agg); stats

,method,epi_mean,epi_std,viol_mean,Tcorr,n,gap_to_oracle
4,rule_based,0.335426,0.000000,146.0,88.541667,2,-1.451755
3,ppo,0.168011,NaN,629.0,34.375000,1,-1.284340
1,nn_mpc,0.005448,0.102030,248.0,88.368056,2,-1.121777
5,sac,-0.014796,NaN,490.0,29.861111,1,-1.101533
0,grey_box_mpc,-0.260353,0.304359,168.0,86.805556,2,-0.855976
6,sindy_mpc,-0.580225,0.196561,433.0,30.902778,2,-0.536104
2,oracle_mpc,-1.116329,NaN,241.0,71.875000,1,0.000000


,method,baseline,metric,n_pairs,mean_diff,ci_low,ci_high,cohen_d,p_value,p_holm,significant
0,grey_box_mpc,rule_based,epi,2,-0.595779,-0.810994,-0.380565,-1.957488,0.5,1.0,False
1,nn_mpc,rule_based,epi,2,-0.329978,-0.402125,-0.257832,-3.234116,0.5,1.0,False
2,oracle_mpc,rule_based,epi,1,-1.451755,-1.451755,-1.451755,NaN,1.0,1.0,False
3,ppo,rule_based,epi,1,-0.167416,-0.167416,-0.167416,NaN,1.0,1.0,False
4,sac,rule_based,epi,1,-0.350222,-0.350222,-0.350222,NaN,1.0,1.0,False
5,sindy_mpc,rule_based,epi,2,-0.915652,-1.054641,-0.776662,-4.658346,0.5,1.0,False


## Парето (EPI vs нарушения) и замкнутые траектории

In [5]:
fig, ax = plt.subplots(figsize=(7, 5))
for meth, g in seeded.groupby("method"):
    x, y = g["violation_steps_total"].mean(), g["epi"].mean()
    ax.scatter(x, y, s=70); ax.annotate(meth, (x, y), fontsize=9, xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("нарушения коридоров, шагов"); ax.set_ylabel("EPI, EUR/m2"); ax.set_title("E3 Парето: EPI vs нарушения"); ax.grid(alpha=.3)
U.save_figure(fig, RES / "figures" / "e3_pareto_epi_violations.png"); plt.close(fig)
if traj0:
    U.plot_rollout_comparison(traj0, RES / "figures", suffix="_e3")
print("saved E3 figures")

saved E3 figures


**Итог E3.** Главная таблица EPI mean±std по регуляторам, разрыв до oracle, Парето-позиция и значимость (Wilcoxon+Holm, bootstrap CI). Дисперсия порождается пересбором train по сидам — претензия протокола к старому `07_multi_seed` снята.